# Maryland Scraper - Azure OpenAI Approach
**AI-driven extraction using GPT-4o**

**Pros:** Flexible, handles varying layouts, demonstrates AI capability  
**Cons:** Costs money (~$0.026 per program), slower (API calls)

In [ ]:
import requests
import json
import time
from typing import Dict
import os

## Azure OpenAI Configuration

In [ ]:
# Azure credentials (use environment variables or set here)
AZURE_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_API_KEY = os.getenv('AZURE_OPENAI_KEY')
AZURE_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt4')
API_VERSION = "2024-12-01-preview"

print(f"Endpoint: {AZURE_ENDPOINT}")
print(f"Deployment: {AZURE_DEPLOYMENT}")
print(f"API Version: {API_VERSION}")

## Helper Functions

In [ ]:
def fetch_page_html(url: str) -> str:
    """Fetch HTML content from URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"   Failed to fetch {url}: {e}")
        return None

In [ ]:
def extract_with_azure_ai(html: str, program_name: str) -> Dict:
    """
    Use Azure OpenAI to extract structured data from HTML
    
    Strategy:
    1. Send HTML to GPT-4o with structured prompt
    2. Request JSON output with specific fields
    3. Parse and return extracted data
    """
    
    # Prepare the prompt
    system_prompt = """You are an expert at extracting structured information from government program web pages.
Extract the following fields from the HTML content:

1. benefits: What the program offers (grants, loans, tax credits, amounts if specified)
2. eligibility: Who can apply (business types, size requirements, location requirements)
3. application_process: How to apply (steps, contacts, required documents)
4. award_amounts: Specific dollar amounts if mentioned (extract exact values)
5. deadlines: Application deadlines or periods (extract exact dates if mentioned)

Return ONLY valid JSON with these exact field names. If a field is not found, use an empty string.
"""
    
    user_prompt = f"""Program Name: {program_name}

HTML Content:
{html[:8000]}

Extract the structured information as JSON."""

    # Prepare API call
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_API_KEY
    }
    
    url = f"{AZURE_ENDPOINT}/openai/deployments/{AZURE_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
    
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1,  # Low temperature for consistency
        "max_tokens": 2000,
        "response_format": {"type": "json_object"}  # Force JSON output
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        content = result['choices'][0]['message']['content']
        
        # Parse the JSON response
        extracted_data = json.loads(content)
        
        return {
            'success': True,
            'data': extracted_data,
            'error': None,
            'tokens_used': result['usage']['total_tokens']
        }
        
    except Exception as e:
        return {
            'success': False,
            'data': {},
            'error': str(e),
            'tokens_used': 0
        }

## Main Scraping Function

In [ ]:
def scrape_program_with_ai(url: str, program_name: str) -> Dict:
    """Scrape a Maryland program detail page using Azure AI"""
    
    print(f"\nAI Scraping: {url}")
    
    # Fetch HTML
    html = fetch_page_html(url)
    if not html:
        return {
            'url': url,
            'program_name': program_name,
            'success': False,
            'error': 'Failed to fetch HTML',
            'data': {},
            'tokens_used': 0
        }
    
    # Extract with AI
    extraction_result = extract_with_azure_ai(html, program_name)
    
    result = {
        'url': url,
        'program_name': program_name,
        **extraction_result
    }
    
    if result['success']:
        print(f"   Extracted successfully ({result['tokens_used']} tokens)")
    else:
        print(f"   Extraction failed: {result['error']}")
    
    return result

## Run Scraper on All Samples

In [ ]:
# Load sample URLs
with open('maryland_sample_urls.json', 'r') as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} sample programs")
print(f"Using Azure deployment: {AZURE_DEPLOYMENT}")
print("=" * 60)

results = []
total_tokens = 0

for i, sample in enumerate(samples, 1):
    print(f"\n[{i}/{len(samples)}] {sample['name']}")
    
    result = scrape_program_with_ai(sample['url'], sample['name'])
    
    # Add metadata from sample
    result['program_type'] = sample['type']
    
    results.append(result)
    total_tokens += result.get('tokens_used', 0)
    
    # Be polite - small delay between requests
    time.sleep(1)

## Save Results

In [ ]:
output_file = 'maryland_azure_results.json'
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 60)
print(f"AI Scraping complete!")
print(f"Results saved to: {output_file}")

# Summary
successful = sum(1 for r in results if r['success'])
print(f"\nSummary:")
print(f"   Successful: {successful}/{len(results)}")
print(f"   Failed: {len(results) - successful}/{len(results)}")
print(f"   Total tokens used: {total_tokens:,}")

# Cost estimate (GPT-4o rough pricing)
estimated_cost = (total_tokens / 1000) * 0.01
print(f"   Estimated cost: ${estimated_cost:.4f}")

## View One Example Result

In [ ]:
# Show first program in detail
program = results[0]
print(f"Program: {program['program_name']}")
print(f"Type: {program['program_type']}")
print(f"Success: {program['success']}")
print(f"Tokens: {program.get('tokens_used', 0)}")
print(f"\nBenefits (first 200 chars):")
print(program['data'].get('benefits', 'N/A')[:200])